# Cell 1: Install & Imports

In [1]:
# Install the correct libraries for the Kaggle Python 3.12 environment
# !pip install transformers==4.35.0 tf-keras 
!pip install transformers==4.39.3 tf-keras xgboost scikit-learn
import os
import warnings
warnings.filterwarnings('ignore')

# Force the current notebook kernel to use Legacy Keras (just in case)
os.environ["TF_USE_LEGACY_KERAS"] = "1"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 79.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 34.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 99.7 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.4.1
    Uninstalling huggingface_hub-1.4.1:
      Successfully uninstalled huggingface_hub-1.4.1
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the follow

Cell 2: Configuration (config.py)
This file controls your whole project. Change paths here only once.

In [2]:
%%writefile config.py
import os

# --- MASTER PATHS ---
BASE_DATA_PATH = "/kaggle/input/datasets/artechie001/cfdf-preprocessed-dataset-non-splited/CelebDF_dataset_split" 
# BASE_DATA_PATH = "/kaggle/input/datasets/aashiqcse/ffpp-unsplited-alltogether/FFPP_dataset_split"
# BASE_DATA_PATH = "/kaggle/input/datasets/artechie001/faceforensics-pngs/FFPP_Splitted_Dataset" 


TRAIN_PATH = os.path.join(BASE_DATA_PATH, "train")
VAL_PATH = os.path.join(BASE_DATA_PATH, "val")
TEST_PATH = os.path.join(BASE_DATA_PATH, "test")

# --- HYPERPARAMETERS ---
IMG_SIZE = (224, 224)
BATCH_SIZE_PER_REPLICA =32
EPOCHS = 20
LEARNING_RATE = 1e-4
DATA_SUBSET_RATIO = 0.99  

# --- OUTPUT FILES (UPDATED FOR 5 MODELS) ---
VIT_WEIGHTS_FILE = "vit_best.weights.h5"
XCP_ATTN_WEIGHTS_FILE = "xcp_attn_best.weights.h5"
XCP_BASE_WEIGHTS_FILE = "xcp_base_best.weights.h5"
EFFB5_WEIGHTS_FILE = "effb5_best.weights.h5"
DENSE_WEIGHTS_FILE = "dense201_best.weights.h5"

Writing config.py


# Cell 3: Utilities (utils.py)
Handles data loading and ImageNet normalization (required for ViT).

In [3]:
%%writefile utils.py
import config
import os
import glob
import tensorflow as tf
from sklearn.utils import shuffle

def get_image_paths(data_path, subset_ratio=config.DATA_SUBSET_RATIO):
    """
    Retrieves and shuffles image paths from the nested directory structure.
    """
    fake_paths = glob.glob(os.path.join(data_path, 'fake', '*', '*.*'))
    real_paths = glob.glob(os.path.join(data_path, 'real', '*', '*.*'))
    
    paths = fake_paths + real_paths
    labels = [1] * len(fake_paths) + [0] * len(real_paths) # 1 for Fake, 0 for Real
    
    paths, labels = shuffle(paths, labels, random_state=42)
    
    if subset_ratio < 1.0:
        limit = int(len(paths) * subset_ratio)
        paths = paths[:limit]
        labels = labels[:limit]
        
    print(f"Loaded {len(paths)} images from {data_path}")
    return paths, labels

def load_and_preprocess_image(path, label):
    """
    Native TensorFlow function to read, decode, resize, and normalize images.
    Crucial for preventing CPU bottlenecks during multi-GPU training.
    """

    img = tf.io.read_file(path)

    img = tf.image.decode_image(img, channels=3, expand_animations=False)

    img = tf.image.resize(img, config.IMG_SIZE)

    img = tf.cast(img, tf.float32) / 255.0
    

    label = tf.one_hot(label, depth=2)
    
    return img, label

def create_tf_dataset(paths, labels, batch_size, is_training=True):
    """
    Builds a highly optimized tf.data.Dataset pipeline.
    Uses AUTOTUNE to dynamically allocate CPU threads for data loading.
    """

    dataset = tf.data.Dataset.from_tensor_slices((paths, labels))
    
    if is_training:

        dataset = dataset.shuffle(buffer_size=len(paths), reshuffle_each_iteration=True)
        

    dataset = dataset.map(load_and_preprocess_image, num_parallel_calls=tf.data.AUTOTUNE)
    

    dataset = dataset.batch(batch_size)
    

    dataset = dataset.prefetch(tf.data.AUTOTUNE)
    
    return dataset

Writing utils.py


# Cell 4: Models (models.py)
Defines your ViXNet (ViT + Xception with Attention).

In [4]:
%%writefile models.py
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"

import tensorflow as tf
from tensorflow.keras import layers, models, Model
from tensorflow.keras.applications import Xception, EfficientNetB5, DenseNet201
from transformers import TFViTModel
from tensorflow.keras import mixed_precision

# Ensure final layers output in float32 for mixed precision stability
FINAL_DTYPE = 'float32'

class ViTWrapper(layers.Layer):
    """Custom layer to wrap the HuggingFace ViT model natively into Keras."""
    def __init__(self, vit_model, **kwargs):
        super().__init__(**kwargs)
        self.vit_model = vit_model
        
    def call(self, inputs):
        x = tf.transpose(inputs, perm=[0, 3, 1, 2])
        outputs = self.vit_model.vit(pixel_values=x)
        return outputs.last_hidden_state[:, 0, :]
        
    def get_config(self):
        return super().get_config()

class AttentionLayer(layers.Layer):
    """Custom Attention Mechanism for feature weighting."""
    def __init__(self, dim, **kwargs):
        super(AttentionLayer, self).__init__(**kwargs)
        self.dim = dim
        
    def get_config(self):
        config = super().get_config()
        config.update({"dim": self.dim})
        return config
        
    def build(self, input_shape):
        self.dense = layers.Dense(self.dim, activation='tanh', use_bias=True)
        self.u_vec = self.add_weight(name='u_vec', shape=(self.dim, 1), initializer='uniform', trainable=True)
        super(AttentionLayer, self).build(input_shape)
        
    def call(self, x):
        u_it = self.dense(x)
        score = tf.matmul(u_it, self.u_vec)
        weights = tf.nn.softmax(score, axis=1)
        return tf.reduce_sum(x * weights, axis=1)

# 1. ViT
def build_vit_classifier(input_shape=(224, 224, 3)):
    inputs = layers.Input(shape=input_shape)
    norm_layer = layers.Normalization(mean=[0.485, 0.456, 0.406], variance=[0.229**2, 0.224**2, 0.225**2])
    x = norm_layer(inputs)
    try:
        vit_model = TFViTModel.from_pretrained('google/vit-base-patch16-224', from_pt=True)
    except:
        vit_model = TFViTModel.from_pretrained('google/vit-base-patch16-224')
    vit_model.trainable = True
    features = ViTWrapper(vit_model, name='vit_features')(x)
    x = layers.Dense(512, activation='relu')(features)
    x = layers.Dropout(0.4)(x)
    outputs = layers.Dense(2, activation='softmax', dtype=FINAL_DTYPE)(x)
    return Model(inputs, outputs, name="ViT_Classifier")

# 2. Xception + Custom Attention
def build_xception_attn_classifier(input_shape=(224, 224, 3)):
    inputs = layers.Input(shape=input_shape)
    x = layers.Rescaling(scale=2.0, offset=-1.0)(inputs) 
    x = layers.GaussianNoise(0.05)(x)
    base = Xception(include_top=False, weights='imagenet', input_tensor=x)
    base.trainable = True
    x = base.output
    x = layers.Reshape((x.shape[1]*x.shape[2], x.shape[3]))(x)
    features = AttentionLayer(dim=512, name='xcp_features')(x)
    x = layers.Dense(512, activation='relu')(features)
    x = layers.Dropout(0.4)(x)
    outputs = layers.Dense(2, activation='softmax', dtype=FINAL_DTYPE)(x)
    return Model(inputs, outputs, name="Xception_Attn_Classifier")

# 3. Base Xception
def build_xception_base_classifier(input_shape=(224, 224, 3)):
    inputs = layers.Input(shape=input_shape)
    x = layers.Rescaling(scale=2.0, offset=-1.0)(inputs) 
    base = Xception(include_top=False, weights='imagenet', input_tensor=x, pooling='avg')
    base.trainable = True
    x = base.output
    x = layers.Dense(512, activation='relu')(x)
    x = layers.Dropout(0.4)(x)
    outputs = layers.Dense(2, activation='softmax', dtype=FINAL_DTYPE)(x)
    return Model(inputs, outputs, name="Xception_Base_Classifier")

# 4. EfficientNetB5
def build_effb5_classifier(input_shape=(224, 224, 3)):
    inputs = layers.Input(shape=input_shape)
    x = layers.Rescaling(scale=255.0)(inputs) 
    
    # --- BULLETPROOF WORKAROUND FOR EFFICIENTNET ---
    # Temporarily drop back to float32 to bypass the internal hardcoded constants bug
    current_policy = mixed_precision.global_policy()
    mixed_precision.set_global_policy('float32')
    
    base = EfficientNetB5(include_top=False, weights='imagenet', pooling='avg')
    
    # Immediately restore the mixed precision policy
    mixed_precision.set_global_policy(current_policy)
    # -----------------------------------------------
    
    # Explicitly force input to float32 before feeding it to the base model
    x = tf.cast(x, tf.float32)
    x = base(x)
    
    x = layers.Dense(512, activation='relu')(x)
    x = layers.Dropout(0.4)(x)
    outputs = layers.Dense(2, activation='softmax', dtype=FINAL_DTYPE)(x)
    return Model(inputs, outputs, name="EfficientNetB5_Classifier")

# 5. DenseNet201
def build_dense201_classifier(input_shape=(224, 224, 3)):
    inputs = layers.Input(shape=input_shape)
    x = layers.Rescaling(scale=255.0)(inputs) 
    
    # Apply standard preprocessing
    x = tf.cast(x, tf.float32)
    x = tf.keras.applications.densenet.preprocess_input(x) 
    
    # Apply the same safety wrapper for DenseNet
    current_policy = mixed_precision.global_policy()
    mixed_precision.set_global_policy('float32')
    
    base = DenseNet201(include_top=False, weights='imagenet', pooling='avg')
    
    mixed_precision.set_global_policy(current_policy)
    
    x = base(x)
    x = layers.Dense(512, activation='relu')(x)
    x = layers.Dropout(0.4)(x)
    outputs = layers.Dense(2, activation='softmax', dtype=FINAL_DTYPE)(x)
    return Model(inputs, outputs, name="DenseNet201_Classifier")

Writing models.py


# Cell 5: Training (train_hybrid.py)
Trains the neural network to get good feature weights.

In [5]:
%%writefile train_separate.py
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"

import config
import gc
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import mixed_precision
from tensorflow.keras.callbacks import ModelCheckpoint, CSVLogger, EarlyStopping, ReduceLROnPlateau
from models import (build_vit_classifier, build_xception_attn_classifier, 
                    build_xception_base_classifier, build_effb5_classifier, 
                    build_dense201_classifier)
from utils import get_image_paths, create_tf_dataset


mixed_precision.set_global_policy('mixed_float16')

strategy = tf.distribute.MirroredStrategy()
print(f"Number of GPUs Synchronized: {strategy.num_replicas_in_sync}")

GLOBAL_BATCH_SIZE = config.BATCH_SIZE_PER_REPLICA * strategy.num_replicas_in_sync
train_paths, train_labels = get_image_paths(config.TRAIN_PATH)
val_paths, val_labels = get_image_paths(config.VAL_PATH)
train_dataset = create_tf_dataset(train_paths, train_labels, GLOBAL_BATCH_SIZE, is_training=True)
val_dataset = create_tf_dataset(val_paths, val_labels, GLOBAL_BATCH_SIZE, is_training=False)

def plot_curve(history, model_name):
    """Generates and saves the training accuracy curves."""
    plt.figure(figsize=(8, 5))
    plt.plot(history.history['accuracy'], label='Train Acc', color='blue')
    plt.plot(history.history['val_accuracy'], label='Val Acc', color='orange')
    plt.title(f'{model_name} - Distributed Accuracy Curve')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.savefig(f"{model_name}_training_plot.png", bbox_inches='tight')
    plt.close()

def train_distributed_model(model_builder, model_name, weight_file_name):
    """Compiles and trains the models across multiple GPUs."""
    print(f"\n{'='*50}\n▶ Starting Distributed Training: {model_name}\n{'='*50}")
    

    with strategy.scope():
        model = model_builder()
        loss_fn = tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1)

        optimizer = tf.keras.optimizers.Adam(config.LEARNING_RATE)
        model.compile(optimizer=optimizer, loss=loss_fn, metrics=['accuracy'])

    callbacks = [
        ModelCheckpoint(weight_file_name, monitor='val_loss', mode='min', save_best_only=True, save_weights_only=True, verbose=1),
        CSVLogger(f"{model_name}_training_log.csv"),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7, verbose=1),
        EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
    ]


    history = model.fit(
        train_dataset, 
        validation_data=val_dataset, 
        epochs=config.EPOCHS, 
        callbacks=callbacks
    )

    plot_curve(history, model_name)
    

    del model
    tf.keras.backend.clear_session()
    gc.collect()
    print(f"\n✔ Finished training {model_name}. GPU Memory flushed.\n")

if __name__ == "__main__":
    train_distributed_model(build_vit_classifier, "ViT", config.VIT_WEIGHTS_FILE)
    train_distributed_model(build_xception_attn_classifier, "Xception_Attn", config.XCP_ATTN_WEIGHTS_FILE)
    train_distributed_model(build_xception_base_classifier, "Xception_Base", config.XCP_BASE_WEIGHTS_FILE)
    train_distributed_model(build_effb5_classifier, "EfficientNetB5", config.EFFB5_WEIGHTS_FILE)
    train_distributed_model(build_dense201_classifier, "DenseNet201", config.DENSE_WEIGHTS_FILE)
    print("All 5 models successfully trained using Dual GPUs!")

Writing train_separate.py


Cell 6: Run Training

In [6]:
!TF_USE_LEGACY_KERAS=1 python train_separate.py

2026-05-15 09:34:56.577722: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778837696.806685      53 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778837696.871740      53 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778837697.373801      53 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778837697.373845      53 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778837697.373849      53 computation_placer.cc:177] computation placer alr

In [7]:
%%writefile ablation_study_separate.py
import config
import numpy as np
import tensorflow as tf 
import itertools
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score, precision_score, recall_score, f1_score
from models import (build_vit_classifier, build_xception_attn_classifier, 
                    build_xception_base_classifier, build_effb5_classifier, 
                    build_dense201_classifier)
from utils import get_image_paths, create_tf_dataset

def print_evaluation_metrics(y_true, y_pred, y_prob, name):
    """
    Calculates and prints performance metrics (AUC, Precision, Recall, F1).
    Ensures comprehensive model evaluation documentation.
    """
    acc = accuracy_score(y_true, y_pred)
    auc = roc_auc_score(y_true, y_prob)
    prec = precision_score(y_true, y_pred)
    rec = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    
    print(f"\n--- {name} ---")
    print(f"Accuracy : {acc*100:.2f}%")
    print(f"AUC      : {auc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall   : {rec:.4f}")
    print(f"F1-Score : {f1:.4f}")
    print("-" * 30)
    print(classification_report(y_true, y_pred, target_names=['Real', 'Fake']))

def run_ablation():
    print("--- FULL ABLATION STUDY: 5 Individual Models + All Pairs ---")
    
    # Sync scope required to load weights gracefully on identical topology
    strategy = tf.distribute.MirroredStrategy()
    GLOBAL_BATCH_SIZE = config.BATCH_SIZE_PER_REPLICA * strategy.num_replicas_in_sync
    
    models_info = {
        "ViT": (build_vit_classifier, config.VIT_WEIGHTS_FILE),
        "Xception_Attn": (build_xception_attn_classifier, config.XCP_ATTN_WEIGHTS_FILE),
        "Xception_Base": (build_xception_base_classifier, config.XCP_BASE_WEIGHTS_FILE),
        "EfficientNetB5": (build_effb5_classifier, config.EFFB5_WEIGHTS_FILE),
        "DenseNet201": (build_dense201_classifier, config.DENSE_WEIGHTS_FILE)
    }
    
    loaded_models = {}
    
    # 1. Load All Models inside strategy scope
    with strategy.scope():
        for name, (builder, weight_file) in models_info.items():
            print(f"Loading {name}...")
            model = builder()
            model.load_weights(weight_file)
            loaded_models[name] = model

    # 2. Load Test Data using high-performance tf.data pipeline
    test_paths, test_labels = get_image_paths(config.TEST_PATH)
    print(f"Evaluating on {len(test_paths)} test images...")
    
    # Set is_training=False so shuffle is disabled; ordering is preserved for evaluation
    test_dataset = create_tf_dataset(test_paths, test_labels, GLOBAL_BATCH_SIZE, is_training=False)
    
    all_probs = {name: [] for name in loaded_models.keys()}
    
    # 3. Predict via native distribution
    for name, model in loaded_models.items():
        print(f"Generating predictions for {name}...")
        # model.predict automatically distributes inference across GPUs
        probs = model.predict(test_dataset, verbose=1)
        all_probs[name] = probs

    # Since we didn't shuffle the dataset, y_true aligns with the initial labels list
    y_true = test_labels
    
    # 4. Generate Reports for Individual Models
    print("\n\n" + "#"*60)
    print("   PART 1: INDIVIDUAL MODEL PERFORMANCE")
    print("#"*60)
    
    for name in loaded_models.keys():
        probs = np.array(all_probs[name])
        preds = np.argmax(probs, axis=1)
        prob_fake = probs[:, 1] # Extract 'Fake' class probability for AUC computation
        print_evaluation_metrics(y_true, preds, prob_fake, f"{name} Only")

    # 5. Generate Reports for Combinations (Pairs of 2)
    print("\n\n" + "#"*60)
    print("   PART 2: LATE FUSION (ALL PAIRS OF 2 MODELS)")
    print("#"*60)
    
    # Generate all pairs dynamically (5 Choose 2 = 10 combinations)
    pairs = list(itertools.combinations(loaded_models.keys(), 2))
    
    for (m1, m2) in pairs:
        probs1 = np.array(all_probs[m1])
        probs2 = np.array(all_probs[m2])
        
        # Late Fusion: Average the probability distributions
        fused_probs = (probs1 + probs2) / 2.0
        fused_preds = np.argmax(fused_probs, axis=1)
        fused_prob_fake = fused_probs[:, 1]
        
        print_evaluation_metrics(y_true, fused_preds, fused_prob_fake, f"FUSION: {m1} + {m2}")

if __name__ == "__main__":
    run_ablation()

Writing ablation_study_separate.py


In [8]:
!python ablation_study_separate.py

2026-05-15 16:36:34.001764: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778862994.026068   23031 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778862994.034062   23031 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778862994.055230   23031 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778862994.055258   23031 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778862994.055262   23031 computation_placer.cc:177] computation placer alr

# For Cross

In [9]:
%%writefile config.py
import os

# --- Cross MASTER PATHS ---
# BASE_DATA_PATH = "/kaggle/input/datasets/artechie001/cfdf-preprocessed-dataset-non-splited/CelebDF_dataset_split" 
# BASE_DATA_PATH = "/kaggle/input/datasets/aashiqcse/ffpp-unsplited-alltogether/FFPP_dataset_split"
BASE_DATA_PATH = "/kaggle/input/datasets/artechie001/faceforensics-pngs/FFPP_Splitted_Dataset" 

TRAIN_PATH = os.path.join(BASE_DATA_PATH, "train")
VAL_PATH = os.path.join(BASE_DATA_PATH, "val")
TEST_PATH = os.path.join(BASE_DATA_PATH, "test")

# --- HYPERPARAMETERS ---
IMG_SIZE = (224, 224)
BATCH_SIZE_PER_REPLICA =32
EPOCHS = 20
LEARNING_RATE = 1e-4
DATA_SUBSET_RATIO = 0.99  

# --- OUTPUT FILES (UPDATED FOR 5 MODELS) ---
VIT_WEIGHTS_FILE = "vit_best.weights.h5"
XCP_ATTN_WEIGHTS_FILE = "xcp_attn_best.weights.h5"
XCP_BASE_WEIGHTS_FILE = "xcp_base_best.weights.h5"
EFFB5_WEIGHTS_FILE = "effb5_best.weights.h5"
DENSE_WEIGHTS_FILE = "dense201_best.weights.h5"

Overwriting config.py


In [10]:
!python ablation_study_separate.py

2026-05-15 16:39:49.516206: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778863189.540699   23392 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778863189.548677   23392 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778863189.568034   23392 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778863189.568084   23392 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778863189.568092   23392 computation_placer.cc:177] computation placer alr